[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/stabilizer/stabilizer_spinors.ipynb)

# Four-Qubit Stabilizer Code with Spinor States

This notebook builds the four-qubit quantum stabilizer code using spinor state vectors in a minimal left ideal, detecting any single-qubit bit flip or phase flip while storing two logical qubits. One syndrome check tests whether flipping all four physical qubits leaves the state vector unchanged, and a second tests whether phase-flipping all four does. Encoded states pass both checks, whereas any single-qubit error flips at least one check outcome so that the corrupted run is rejected, leaving undetected logical errors only when at least two qubits suffer errors in the same run.

An algebra with four positive-square and four negative-square generators has 256 basis blades, matching the real linear operators on sixteen real four-qubit spinor components. Multiplying four commuting idempotent factors `(1 + blade) / 2` isolates a sixteen-dimensional minimal left ideal closed under left multiplication, and choosing sixteen basis blades in a four-generator subalgebra selects spinor representatives on which operators act via `readout(Full * embedding)`. Computing expectations and overlaps then requires an explicit positive-definite inner product `pairing(states, action(operators, states))` built from reverse conjugation and fixed reference blades of the chosen ideal, in contrast to [the density-operator notebook](stabilizer.ipynb), where states and observables share one space and expectations use `scalar_product` directly.

In Pauli stabilizer and Dirac ket notation ([Gottesman, 1997](https://arxiv.org/abs/quant-ph/9705052)), the $[[4, 2, 2]]$ code subspace of $\mathbb{R}^{16} \subset (\mathbb{C}^2)^{\otimes 4}$ is the joint $+1$ eigenspace of $S_1 = X \otimes X \otimes X \otimes X$ and $S_2 = Z \otimes Z \otimes Z \otimes Z$, with logical operators $\bar X_1 = X \otimes X \otimes I \otimes I$, $\bar Z_1 = Z \otimes I \otimes Z \otimes I$, $\bar X_2 = X \otimes I \otimes X \otimes I$, and $\bar Z_2 = Z \otimes Z \otimes I \otimes I$. Operators act from the left on a column ket $|\psi\rangle$ as $|\psi'\rangle = U|\psi\rangle$, expectations evaluate the bra-ket bilinear form $\langle\psi|O|\psi\rangle / \langle\psi|\psi\rangle$, and post-selected state fidelity reads $|\langle\psi|\psi'\rangle|^2 / \langle\psi'|P_{\mathrm{code}}|\psi'\rangle$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from functools import reduce
from operator import mul

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure
from examples.quantum.stabilizer import render

ga = Algebra("a+b+c+d+e-f-g-h-")
mv = NumpyContext(ga).multivector
exact = ga.exact.multivector

Scalar = ga.gatype.scalar()                                                    # Scalar+
Full = ga.gatype.full()                                                        # Full+
State = Full.derive.within(Algebra("b+c+d+h-"))                                # Full+

# Four commuting square-one elements build four projectors whose product generates a sixteen-dimensional left ideal.
ideal = (1 + exact.a) * (1 + exact.be) * (1 + exact.cf) * (1 + exact.dg) / 16  # [] Full+
# Multiplying the sixteen State blades into the left ideal chooses a spinor basis.
embedding = State * ideal                                                      # [] Full+ <- State+
# Scaling by 16 undoes the ideal normalization when reading spinor representatives back out.
readout = (16 * Full).cast(State)                                              # [] State+ <- Full+
action = readout(Full * embedding)                                             # [] State+ <- (Full+, State+)
# Reference blades efgh and cdeh make the State basis orthonormal under a positive-definite overlap pairing.
pairing = 16 * exact.cdeh.scalar_product(embedding.reverse() * exact.efgh * embedding)  # [] Scalar+ <- (State+, State+)

# Bit flips and phase flips square to +1, anticommute on the same qubit, and commute across distinct qubits.
flips = stack([mv.b, -mv.bce, -mv.bcdef, -mv.ah])                              # [4] Full+
phase_flips = stack([mv.be, mv.cf, mv.dg, -mv.abcdefg])                        # [4] Full+
# Each qubit undergoes the identity, a bit flip, a phase flip, or their product.
errors = stack([mv.scalar().broadcast_to(flips.shape), flips, phase_flips, flips * phase_flips], axis=-1)  # [4, 4] Full+
# All four phase flips leave the scalar representative fixed as the all-zero spinor.
ground = mv.scalar().cast(State)                                               # [] State+

## Stabilizer checks and logical operators

Each stabilizer check multiplies four single-qubit blades, one per physical qubit. Because a qubit's bit flip and phase flip anticommute and the two checks overlap on all four qubits, an even number of anticommutations makes the two checks commute. Multiplying the half-projectors `(1 + checks * outcomes) / 2` across all four sign patterns builds four orthogonal syndrome projectors that sum to the identity, with the `(+1, +1)` projector defining the code subspace. Each logical flip and phase flip acts on a pair of physical qubits, overlapping each check on two qubits so that all logical operators commute with both checks.

In [ ]:
# One check flips all four qubits and the other phase-flips all four.
checks = stack([reduce(mul, flips), reduce(mul, phase_flips)])                 # [2] Full+
# Pairing both checks with all four sign patterns builds four orthogonal syndrome projectors, with (+1, +1) as the code.
outcomes = np.array([[1, 1], [1, -1], [-1, 1], [-1, -1]])                      # [4, 2]
halves = (1 + checks * outcomes) / 2                                           # [4, 2] Full+
projectors = halves[:, 0] * halves[:, 1]                                       # [4] Full+
code = projectors[0]                                                           # [] Full+

# Pairwise bit flips and phase flips commute with both checks and act on the two encoded qubits.
logical_flips = stack([flips[0] * flips[1], flips[0] * flips[2]])              # [2] Full+
logical_phase_flips = stack([phase_flips[0] * phase_flips[2], phase_flips[0] * phase_flips[1]])  # [2] Full+
logical_operators = stack([logical_flips, logical_phase_flips], axis=-1)       # [2, 2] Full+

## Encoding logical superpositions

Acting with the code projector on the all-zero spinor prepares the logical zero state, and exponentiating each logical qubit's flip times its phase flip rotates the spinor state space continuously from 0 toward 1. Because each generator squares to `-1` and preserves the spinor inner product `pairing`, left multiplication via `action` turns the state at unit norm without requiring grade-2 bivectors. Every logical turn commutes with both stabilizer checks, so the encoded spinor keeps check expectations equal to `+1` while the logical flip and phase-flip expectations recover the sine and cosine of each angle.

In [ ]:
from examples.quantum.stabilizer.scenarios import angles

In [ ]:
def expectation(operators: Full, states: State) -> Scalar:
    """The spinor pairing evaluates the normalized expectation of `operators` in `states` (`[...] Scalar+`)."""
    return pairing(states, action(operators, states)) / pairing(states, states)  # [...] Scalar+


def turned_state(
    initial_state: State,
    qubit_flips: Full,
    qubit_phase_flips: Full,
    angles: np.ndarray,
) -> State:
    """Left action by the exponential of each qubit's flip times its phase flip turns `initial_state` from 0 toward 1 by `angles` (`[...] State+`)."""
    turns = ((qubit_flips * qubit_phase_flips) * (angles / 2)).exp()           # [..., 2] Full+
    return action(turns[..., 0] * turns[..., 1], initial_state)                # [...] State+


projected_ground = action(code, ground)                                        # [] State+
code_ground = projected_ground / pairing(projected_ground, projected_ground).square_root()  # [] State+
stored_state = turned_state(code_ground, logical_flips, logical_phase_flips, angles)  # [] State+

check_readings = expectation(checks, stored_state[..., None])                  # [2] Scalar+
logical_readings = expectation(logical_operators, stored_state[..., None, None])  # [2, 2] Scalar+

print("checks:", check_readings.to_array())
print("flip and phase flip of each encoded qubit:\n", logical_readings.to_array())
print("sine and cosine of each angle:\n", np.stack([np.sin(angles), np.cos(angles)], axis=-1))

## Detecting single-qubit errors

A bit flip on any single qubit anticommutes with the four-qubit phase-flip check, a phase flip anticommutes with the bit-flip check, and a combined flip-and-phase error anticommutes with both. Applying each single-qubit error to the encoded spinor via `action` therefore flips at least one check expectation from `+1` to `-1` in every non-trivial column of the syndrome table. Because all four qubits produce the same syndrome for a given error type, the two checks flag corrupted runs for rejection while two-qubit errors such as a pair of bit flips can still commute with both checks.

In [ ]:
readings = expectation(checks, action(errors, stored_state)[..., None])        # [4, 4, 2] Scalar+
print(render.syndrome_table(readings))

## Acceptance and fidelity under noise

Under independent depolarizing noise, each physical qubit either stays untouched or suffers one of the three non-trivial errors with equal probability, producing 256 four-qubit error patterns weighted by their joint probabilities. On spinor states, `pairing(corrupted_states, action(code, corrupted_states))` gives the probability that both checks pass, while `pairing(stored_state, corrupted_states).squared()` measures squared overlap with the intended state. Because rejected states are orthogonal to the code subspace containing `stored_state`, dividing the mean squared overlap by the acceptance probability yields the fidelity conditional on passing both checks.

In matrix notation, the minimal left ideal identifies spinors with column vectors $|\psi\rangle \in \mathbb{R}^{16}$ acted on by four-fold Kronecker products of $I$, $X$, $Z$, and $XZ$, so post-selected fidelity evaluates $\sum_m p_m |\langle\psi|E_m|\psi\rangle|^2 / \sum_m p_m \langle\psi|E_m^\dagger P_{\mathrm{code}} E_m|\psi\rangle$.

In [ ]:
from examples.quantum.stabilizer.scenarios import n_rates, rates

In [ ]:
def error_patterns(n_qubits: int, rates: np.ndarray) -> tuple[Full, np.ndarray]:
    """Independent depolarizing noise on `n_qubits` qubits pairs every multi-qubit error blade (`[...] Full+`) with its probability (`[...]`) across `rates`."""
    kinds = np.indices((errors.shape[-1],) * n_qubits)                         # [n_qubits, ...]
    chances = np.stack([1 - rates, rates / 3, rates / 3, rates / 3], axis=-1)  # [..., 4]
    patterns = reduce(mul, (errors[qubit, kind] for qubit, kind in enumerate(kinds)))  # [...] Full+
    weights = reduce(mul, (chances[:, kind] for kind in kinds))                # [...]
    return patterns, weights


def pattern_average(values: Scalar, weights: np.ndarray) -> Scalar:
    """Weighting `values` by `weights` sums over the trailing error-pattern axes to give the mean at each noise rate (`[...] Scalar+`)."""
    return (values * weights).sum(axis=tuple(range(1 - weights.ndim, 0)))      # [...] Scalar+

In [ ]:
def code_performance(stored_state: State, rates: np.ndarray) -> tuple[Scalar, Scalar]:
    """Depolarizing noise on all four physical qubits gives the fraction of runs passing both checks and the conditional infidelity of accepted runs (`[...] Scalar+`)."""
    patterns, weights = error_patterns(errors.shape[0], rates)
    corrupted_states = action(patterns, stored_state)                          # [4, 4, 4, 4] State+
    # Projecting onto the code subspace measures the probability that both checks read +1.
    acceptance = pairing(corrupted_states, action(code, corrupted_states))     # [4, 4, 4, 4] Scalar+
    # Rejected spinors are orthogonal to the code subspace containing stored_state.
    overlap_squared = pairing(stored_state, corrupted_states).squared()        # [4, 4, 4, 4] Scalar+
    kept_fraction = pattern_average(acceptance, weights)                       # [...] Scalar+
    return kept_fraction, 1 - pattern_average(overlap_squared, weights) / kept_fraction  # [...] Scalar+, Scalar+


def bare_infidelity(angles: np.ndarray, rates: np.ndarray) -> Scalar:
    """Depolarizing noise on two unencoded qubits at `angles` gives their unconditional infidelity at each rate (`[...] Scalar+`)."""
    n_bare = angles.shape[-1]
    bare_state = turned_state(ground, flips[:n_bare], phase_flips[:n_bare], angles)  # [] State+
    patterns, weights = error_patterns(n_bare, rates)
    overlap_squared = pairing(bare_state, action(patterns, bare_state)).squared()  # [4, 4] Scalar+
    return 1 - pattern_average(overlap_squared, weights)                       # [...] Scalar+


kept_fraction, encoded_errors = code_performance(stored_state, rates)          # [n_rates] Scalar+, Scalar+
bare_errors = bare_infidelity(angles, rates)                                   # [n_rates] Scalar+

save_figure(render.noise_figure(rates, kept_fraction, encoded_errors, bare_errors), "stabilizer_noise");

The blue curve shows the conditional infidelity of accepted encoded runs, which grows quadratically at small per-qubit error rates because every single-qubit error is rejected, whereas the red curve for two unencoded qubits grows linearly from the origin. The dashed grey curve tracks the fraction of runs passing both stabilizer checks, falling from `1` as single-qubit rejections become frequent.

In [ ]:
# checks
assert (projectors.sum(axis=0) - 1).scalar_norm_squared() < 1e-20
assert ((check_readings - 1).abs() < 1e-12).all()
expected_logical = np.stack([np.sin(angles), np.cos(angles)], axis=-1)
assert ((logical_readings - expected_logical).abs() < 1e-12).all()
assert ((readings[:, 0] - 1).abs() < 1e-12).all()
assert (readings[:, 1:] < 0).any(axis=-1).all()
# Doubling a small error rate quadruples conditional infidelity and doubles bare-qubit infidelity.
assert (encoded_errors[2] / encoded_errors[1] - 4).abs() < 0.2
assert (bare_errors[2] / bare_errors[1] - 2).abs() < 0.1